# AmsiNet - Steps 1 + 2 in ONE notebook

Run the cells from top to bottom. **Part 1** downloads BSTD, builds the page > block > line > word index and prints dataset statistics (slow part: the 17 GB download). **Part 2** cuts word crops, trains the flat baselines and measures backbone efficiency.

Use a Colab Pro **A100** runtime. All results and the index are saved to `MyDrive/amsinet/`, so a disconnect does not lose them.


# PART 1 - Get the Bharat Scene Text Dataset (BSTD) and build the page > block > line > word hierarchy

**Runtime:** Colab Pro, CPU is enough for this notebook (GPU is needed from Step 2).

What this notebook does
1. Mounts Google Drive and downloads BSTD detection data (about 17 GB zip) from the dataset authors' Google Drive link.
2. Reads the annotations (`BSTD_v17.57.json`: polygon, text, `script_language` per word).
3. Maps the 12 language tags to scripts (Hindi/Marathi share Devanagari, Assamese/Bengali share one script).
4. Builds a geometry-derived hierarchy (page > block > line > word) and a validation split made by image.
5. Prints dataset statistics you can use in the paper (mixed-script rates etc.) and saves an index file to Drive.

Dataset: Bharat Scene Text Dataset, https://github.com/Bhashini-IITJ/BharatSceneTextDataset (images CC BY-SA 4.0).

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT  = '/content/drive/MyDrive/amsinet'     # outputs persist here
LOCAL = '/content/bstd'                      # big files live on the fast local disk
os.makedirs(f'{ROOT}/data', exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)
!pip -q install gdown

## 2. Download BSTD detection data
If the direct download fails with a "quota exceeded" message (common for large Google Drive files), open the link from the dataset page in your browser, use **Add shortcut to Drive / Make a copy**, then set `DET_ZIP` below to the copied file in `/content/drive/MyDrive/...` and skip the download line.

In [ ]:
import gdown, glob, subprocess

DET_ID = '1S7KUYfB-lQvbu6GtvZxxDPOD5ZZ080M0'      # detection.zip (about 17 GB), from the BSTD GitHub page
DET_ZIP = f'{LOCAL}/detection.zip'

if not os.path.exists(DET_ZIP):
    gdown.download(id=DET_ID, output=DET_ZIP, quiet=False)

# unzip once (takes several minutes)
if not glob.glob(f'{LOCAL}/**/BSTD_v17.57.json', recursive=True):
    subprocess.run(['unzip', '-q', '-o', DET_ZIP, '-d', LOCAL], check=True)

ANN_PATH = glob.glob(f'{LOCAL}/**/BSTD_v17.57.json', recursive=True)[0]
IMG_ROOT = os.path.dirname(ANN_PATH)
print('annotations:', ANN_PATH)
print('image root :', IMG_ROOT)

## 3. Inspect the raw annotation values (always check before trusting label names)

In [ ]:
import json
from collections import Counter

with open(ANN_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

print('images in annotation file:', len(raw))
tags = Counter()
splits = Counter()
for k, v in raw.items():
    splits[v.get('split')] += 1
    for p in v.get('annotations', {}).values():
        tags[str(p.get('script_language')).strip().lower()] += 1
print('split values:', dict(splits))
print('script_language values:')
for t, n in tags.most_common():
    print(f'  {t!r:20} {n}')

## 4. Label mapping and hierarchy builder
`LANG2SCRIPT` merges languages that share a script (the BSTD authors do the same for their end-to-end evaluation). Words with other tags (for example Urdu or Meitei, which BSTD marks as "Others") are kept in the index with script `None` and ignored in training and statistics.

In [ ]:
import numpy as np, random

LANG2SCRIPT = {
    'english': 'Latin',
    'hindi': 'Devanagari', 'marathi': 'Devanagari',
    'bengali': 'Bengali-Assamese', 'assamese': 'Bengali-Assamese',
    'gujarati': 'Gujarati', 'kannada': 'Kannada', 'malayalam': 'Malayalam',
    'odia': 'Odia', 'oriya': 'Odia', 'punjabi': 'Gurmukhi', 'panjabi': 'Gurmukhi',
    'tamil': 'Tamil', 'telugu': 'Telugu',
}
SCRIPTS = sorted(set(LANG2SCRIPT.values()))
LANGS = sorted(set(k for k in LANG2SCRIPT if k not in ('oriya', 'panjabi')))
print('script classes (%d):' % len(SCRIPTS), SCRIPTS)

def norm_lang(x):
    return str(x).strip().lower()

def poly_to_bbox(coords):
    xs = [p[0] for p in coords]; ys = [p[1] for p in coords]
    return [float(min(xs)), float(min(ys)), float(max(xs)), float(max(ys))]   # x1, y1, x2, y2

def _union(boxes):
    b = np.array(boxes, dtype=float)
    return [b[:, 0].min(), b[:, 1].min(), b[:, 2].max(), b[:, 3].max()]

def group_lines(words):
    # Greedy line grouping: a word joins a line if it is vertically aligned and horizontally close.
    # Heuristic for roughly horizontal text; rotated text ends up in its own groups.
    order = sorted(range(len(words)), key=lambda i: (words[i]['bbox'][1] + words[i]['bbox'][3]) / 2)
    lines = []                                                  # each: {'ids': [...], 'box': [...]}
    for i in order:
        x1, y1, x2, y2 = words[i]['bbox']
        h = max(y2 - y1, 1.0); yc = (y1 + y2) / 2
        best, best_gap = None, None
        for li, ln in enumerate(lines):
            lx1, ly1, lx2, ly2 = ln['box']
            lh = max(ly2 - ly1, 1.0); lyc = (ly1 + ly2) / 2
            ref = max(h, lh)
            if abs(yc - lyc) > 0.5 * ref:
                continue
            gap = max(x1 - lx2, lx1 - x2, 0.0)
            if gap <= 2.5 * ref and (best is None or gap < best_gap):
                best, best_gap = li, gap
        if best is None:
            lines.append({'ids': [i], 'box': [x1, y1, x2, y2]})
        else:
            lines[best]['ids'].append(i)
            lines[best]['box'] = _union([lines[best]['box'], [x1, y1, x2, y2]])
    return lines

def box_gap(a, b):
    dx = max(a[0] - b[2], b[0] - a[2], 0.0)
    dy = max(a[1] - b[3], b[1] - a[3], 0.0)
    return (dx ** 2 + dy ** 2) ** 0.5

def group_blocks(lines, factor=1.5):
    # Merge lines whose boxes are closer than factor x median line height (union-find).
    n = len(lines)
    if n == 0:
        return []
    heights = [max(l['box'][3] - l['box'][1], 1.0) for l in lines]
    thr = factor * float(np.median(heights))
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for i in range(n):
        for j in range(i + 1, n):
            if box_gap(lines[i]['box'], lines[j]['box']) <= thr:
                parent[find(i)] = find(j)
    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return list(groups.values())                                # list of lists of line indices

def build_index(raw, val_frac=0.10, seed=42):
    rng = random.Random(seed)
    train_keys = [k for k, v in raw.items() if 'train' in str(v.get('split')).lower()]
    val_keys = set(rng.sample(train_keys, int(round(val_frac * len(train_keys)))))   # split by IMAGE, never by word
    index = []
    for key, rec in raw.items():
        words = []
        for p in rec.get('annotations', {}).values():
            coords = p.get('coordinates')
            if not coords:
                continue
            lang = norm_lang(p.get('script_language'))
            words.append({'bbox': poly_to_bbox(coords), 'text': p.get('text', ''),
                          'lang': lang, 'script': LANG2SCRIPT.get(lang)})
        if not words:
            continue
        lines = group_lines(words)
        blocks = group_blocks(lines)
        for li, ln in enumerate(lines):
            for wi in ln['ids']:
                words[wi]['line'] = li
        for bi, line_ids in enumerate(blocks):
            for li in line_ids:
                for wi in lines[li]['ids']:
                    words[wi]['block'] = bi
        split = 'val' if key in val_keys else ('train' if 'train' in str(rec.get('split')).lower() else 'test')
        index.append({'id': key, 'image': rec.get('image_name'), 'split': split,
                      'n_lines': len(lines), 'n_blocks': len(blocks), 'words': words})
    return index

def soft_label(words):
    # Distribution of scripts over the words of a region = coarse-level target y_l(r) from Section 3.1
    c = Counter(w['script'] for w in words if w['script'])
    tot = sum(c.values())
    return {s: c[s] / tot for s in c} if tot else {}

In [ ]:
index = build_index(raw)
print('images in index:', len(index))
print(Counter(r['split'] for r in index))

## 5. Dataset statistics for the paper (copy these numbers into Section 4 of the manuscript)

In [ ]:
import statistics as st

def dataset_stats(index):
    out = {}
    for sp in ('train', 'val', 'test', 'all'):
        recs = [r for r in index if sp == 'all' or r['split'] == sp]
        ws = [w for r in recs for w in r['words'] if w['script']]
        per_img = [len({w['script'] for w in r['words'] if w['script']}) for r in recs]
        mixed = [n >= 2 for n in per_img if n >= 1]
        # words whose script differs from the image's dominant script
        minority = tot = 0
        for r in recs:
            c = Counter(w['script'] for w in r['words'] if w['script'])
            if not c: continue
            dom = c.most_common(1)[0][0]
            tot += sum(c.values()); minority += sum(v for s, v in c.items() if s != dom)
        out[sp] = {
            'images': len(recs), 'words_labelled': len(ws),
            'words_per_script': dict(Counter(w['script'] for w in ws)),
            'words_per_language': dict(Counter(w['lang'] for w in ws)),
            'images_with_2plus_scripts_pct': round(100 * sum(mixed) / max(len(mixed), 1), 1),
            'mean_scripts_per_image': round(st.mean(per_img), 2) if per_img else 0,
            'words_not_in_dominant_script_pct': round(100 * minority / max(tot, 1), 1),
            'mean_words_per_image': round(len(ws) / max(len(recs), 1), 1),
            'mean_lines_per_image': round(st.mean(r['n_lines'] for r in recs), 1) if recs else 0,
            'mean_blocks_per_image': round(st.mean(r['n_blocks'] for r in recs), 1) if recs else 0,
        }
    return out

S = dataset_stats(index)
for sp in ('train', 'val', 'test', 'all'):
    print('=' * 8, sp)
    for k, v in S[sp].items():
        print(f'  {k}: {v}')

## 6. Save the index to Drive and look at one mixed-script example

In [ ]:
with open(f'{ROOT}/data/bstd_index.json', 'w', encoding='utf-8') as f:
    json.dump({'img_root': IMG_ROOT, 'scripts': SCRIPTS, 'index': index}, f, ensure_ascii=False)
with open(f'{ROOT}/data/bstd_stats.json', 'w', encoding='utf-8') as f:
    json.dump(S, f, indent=2, ensure_ascii=False)
print('saved to', f'{ROOT}/data')

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image

mixed = [r for r in index if len({w['script'] for w in r['words'] if w['script']}) >= 3]
rec = random.Random(0).choice(mixed if mixed else index)
img = Image.open(os.path.join(IMG_ROOT, rec['image'])).convert('RGB')
fig, ax = plt.subplots(figsize=(10, 7)); ax.imshow(img); ax.axis('off')
cmap = plt.get_cmap('tab10')
for w in rec['words']:
    x1, y1, x2, y2 = w['bbox']
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=1.5,
                                   edgecolor=cmap(w.get('block', 0) % 10)))
    ax.text(x1, y1, str(w['script'])[:3], color='yellow', fontsize=7, backgroundcolor='black')
ax.set_title(f"{rec['id']} | blocks={rec['n_blocks']} lines={rec['n_lines']} (box colour = block)")
plt.show()

# PART 2 - flat baselines on BSTD (word-level script identification)

**Runtime:** Colab Pro with a GPU (A100 if available, otherwise T4). Run Step 1 first; it creates `bstd_index.json` on your Drive.

What this notebook does
1. Cuts every labelled word out of the BSTD images once (letterboxed to 224x224) and stores the crops on the local disk.
2. Fine-tunes flat (single-agent, non-hierarchical) classifiers over the 10 scripts, one run per seed, with validation-based model selection and a single test evaluation.
3. Reports accuracy and macro-F1 as mean and 95% confidence interval over seeds, plus per-script F1 and confusion matrices.
4. Measures parameters, FLOPs and latency of the backbones you plan to use as agents (this fills Table 2 of the manuscript with measured values).

Protocol notes (write these in Section 4 of the paper)
- Splits are by **image** (train / val / test), so words from one photo never appear in two splits.
- The BSTD authors' published 12-way numbers use a different, class-balanced protocol on 12 languages. Do not compare with them directly unless you also run their `script_identification.zip` protocol.
- No horizontal flips: mirroring changes glyph shapes and therefore the script.

In [ ]:
# Part 2 setup (Drive is already mounted in Part 1)
!pip -q install timm gdown
import torch, timm
print('torch', torch.__version__, '| timm', timm.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch runtime to GPU)')

In [ ]:
# 2. Configuration and the Step-1 index
import json, os, glob, random, time, math
import numpy as np, pandas as pd

ROOT  = '/content/drive/MyDrive/amsinet'
LOCAL = '/content/bstd'

CFG = dict(
    MODELS = ['vit_base_patch16_224.augreg_in21k'],   # add e.g. 'convnext_tiny.fb_in22k' to get flat baselines with the agent backbones
    SEEDS  = [0],                                     # use [0, 1, 2, 3, 4] for the final paper numbers
    EPOCHS = 5, BATCH = 64, LR = 5e-5, WD = 0.05, IMG = 224, WORKERS = 8,
    LABEL_SMOOTHING = 0.1, CLASS_BALANCED_LOSS = False,
    APPLY_EXIF = True,                                # honour photo orientation tags (cv2 and browsers do)
    CROP_PAD = 0.04,                                  # context margin around each word box
)

with open(f'{ROOT}/data/bstd_index.json', 'r', encoding='utf-8') as f:
    D = json.load(f)
IMG_ROOT, SCRIPTS, INDEX = D['img_root'], D['scripts'], D['index']
LABEL = {s: i for i, s in enumerate(SCRIPTS)}
print(len(INDEX), 'images |', len(SCRIPTS), 'scripts:', SCRIPTS)

## 2b. Restore data in a fresh runtime
Colab gives each notebook its own temporary disk, so the images from Step 1 are not here. This cell (a) restores the finished crops from a tar file on your Drive if you saved one earlier, otherwise (b) downloads and unzips `detection.zip` again (about 17 GB). After the crops are extracted, the next cells save them to Drive as one tar file so later sessions skip the big download.

In [ ]:
import gdown, subprocess
CROPS = f"{LOCAL}/crops{CFG['IMG']}"
CROPS_TAR = f"{ROOT}/data/crops{CFG['IMG']}.tar"
os.makedirs(LOCAL, exist_ok=True)

if not os.path.isdir(CROPS) and os.path.exists(CROPS_TAR):
    print('restoring crops from Drive ...')
    subprocess.run(['tar', '-xf', CROPS_TAR, '-C', LOCAL], check=True)

if not os.path.isdir(CROPS) and not os.path.isdir(IMG_ROOT):
    print('downloading detection.zip (about 17 GB) ...')
    DET_ZIP = f'{LOCAL}/detection.zip'
    if not os.path.exists(DET_ZIP):
        gdown.download(id='1S7KUYfB-lQvbu6GtvZxxDPOD5ZZ080M0', output=DET_ZIP, quiet=False)
    subprocess.run(['unzip', '-q', '-o', DET_ZIP, '-d', LOCAL], check=True)
    IMG_ROOT = os.path.dirname(glob.glob(f'{LOCAL}/**/BSTD_v17.57.json', recursive=True)[0])
print('crops present:', os.path.isdir(CROPS), '| images present:', os.path.isdir(IMG_ROOT), '| IMG_ROOT =', IMG_ROOT)

## 3. Cut out the word crops (one-time, about 10-20 minutes; resumable)

In [ ]:
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None   # BSTD images come from the dataset authors; a few are huge panoramas
from concurrent.futures import ThreadPoolExecutor

CROPS = f"{LOCAL}/crops{CFG['IMG']}"
FILL = (124, 116, 104)

def letterbox(im, size):
    w, h = im.size
    s = size / max(w, h)
    nw, nh = max(1, int(round(w * s))), max(1, int(round(h * s)))
    im = im.resize((nw, nh), Image.BICUBIC)
    canvas = Image.new('RGB', (size, size), FILL)
    canvas.paste(im, ((size - nw) // 2, (size - nh) // 2))
    return canvas

def extract_one(rec):
    rows, im = [], None
    for k, w in enumerate(rec['words']):
        if not w['script']:
            continue
        d = f"{CROPS}/{rec['split']}/{w['script']}"
        os.makedirs(d, exist_ok=True)
        p = f"{d}/{rec['id']}_{k}.jpg"
        if not os.path.exists(p):
            if im is None:
                im = Image.open(os.path.join(IMG_ROOT, rec['image']))
                if CFG['APPLY_EXIF']:
                    im = ImageOps.exif_transpose(im)
                im = im.convert('RGB')
            x1, y1, x2, y2 = w['bbox']
            if (x2 - x1) < 4 or (y2 - y1) < 4:                       # ignore degenerate boxes
                continue
            px, py = (x2 - x1) * CFG['CROP_PAD'], (y2 - y1) * CFG['CROP_PAD']
            box = (max(0, int(x1 - px)), max(0, int(y1 - py)), min(im.width, int(math.ceil(x2 + px))), min(im.height, int(math.ceil(y2 + py))))
            if box[2] - box[0] < 2 or box[3] - box[1] < 2:
                continue
            letterbox(im.crop(box), CFG['IMG']).save(p, quality=95)
        rows.append((p, w['script'], w['lang'], rec['split'], rec['id'], k))
    return rows

def extract_all(index, workers=8):
    out = []
    with ThreadPoolExecutor(workers) as ex:
        for i, rows in enumerate(ex.map(extract_one, index)):
            out.extend(rows)
            if (i + 1) % 500 == 0:
                print(f'{i + 1}/{len(index)} images done, {len(out)} crops')
    return pd.DataFrame(out, columns=['path', 'script', 'lang', 'split', 'image_id', 'word_idx'])

In [ ]:
META = extract_all(INDEX)
META.to_csv(f'{ROOT}/data/crops_meta.csv', index=False)
print(META.groupby('split').size())
print(META.groupby(['split', 'script']).size().unstack(0).fillna(0).astype(int))

In [ ]:
# Save the crops to Drive as ONE file (fast to restore in any later session)
if not os.path.exists(CROPS_TAR):
    subprocess.run(['tar', '-cf', CROPS_TAR, '-C', LOCAL, f"crops{CFG['IMG']}"], check=True)
print('tar on Drive:', os.path.exists(CROPS_TAR), round(os.path.getsize(CROPS_TAR) / 1e9, 2), 'GB')

### Visual check (do this before training): the crops should show single words. If many look like background or rotated nonsense, set `APPLY_EXIF` to the other value, delete the crops folder and re-extract.

In [ ]:
import matplotlib.pyplot as plt
sm = META.sample(24, random_state=0)
fig, axs = plt.subplots(4, 6, figsize=(12, 8))
for ax, (_, r) in zip(axs.ravel(), sm.iterrows()):
    ax.imshow(Image.open(r.path)); ax.set_title(r.script[:12], fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Evaluation helpers (no GPU code here)

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix
from scipy import stats

def metrics(P, Y):
    pred = P.argmax(1)
    labs = list(range(len(SCRIPTS)))
    return {'acc': float((pred == Y).mean()),
            'macro_f1': float(f1_score(Y, pred, average='macro', labels=labs, zero_division=0)),
            'per_class_f1': f1_score(Y, pred, average=None, labels=labs, zero_division=0).tolist(),
            'confusion': confusion_matrix(Y, pred, labels=labs).tolist()}

def mean_ci(values, level=0.95):
    v = np.asarray(values, dtype=float)
    if len(v) < 2:
        return float(v.mean()), float('nan')
    half = stats.t.ppf(0.5 + level / 2, len(v) - 1) * v.std(ddof=1) / math.sqrt(len(v))
    return float(v.mean()), float(half)

def aggregate(results_dir, model_short):
    files = sorted(glob.glob(f'{results_dir}/{model_short}_seed*.json'))
    R = [json.load(open(f)) for f in files]
    if not R:
        return None
    row = {'model': model_short, 'seeds': len(R)}
    for split in ('val', 'test'):
        for m in ('acc', 'macro_f1'):
            mu, ci = mean_ci([r[split][m] for r in R])
            row[f'{split}_{m}'] = f'{100 * mu:.2f} +/- {100 * ci:.2f}' if ci == ci else f'{100 * mu:.2f}'
    pc = np.array([r['test']['per_class_f1'] for r in R])
    return row, pd.Series(100 * pc.mean(0), index=SCRIPTS, name=model_short)

## 5. Training (flat baseline: one classifier for all words)

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

DEV = 'cuda'
os.makedirs(f'{ROOT}/results', exist_ok=True); os.makedirs(f'{ROOT}/ckpt', exist_ok=True)

class Crops(Dataset):
    def __init__(self, df, tf):
        self.paths = df['path'].tolist(); self.labels = [LABEL[s] for s in df['script']]; self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert('RGB')), self.labels[i]

def make_tfs(model):
    cfg = timm.data.resolve_data_config({}, model=model)
    norm = T.Normalize(cfg['mean'], cfg['std'])
    train = T.Compose([T.RandomAffine(5, translate=(0.03, 0.03), scale=(0.9, 1.1), fill=FILL),
                       T.ColorJitter(0.3, 0.3, 0.2), T.ToTensor(), norm])      # no flips
    test = T.Compose([T.ToTensor(), norm])
    return train, test

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        with torch.autocast('cuda', dtype=torch.float16):
            lo = model(x.to(DEV, non_blocking=True))
        P.append(lo.float().softmax(1).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def run(model_name, seed):
    short = model_name.split('.')[0]
    out = f'{ROOT}/results/{short}_seed{seed}.json'
    if os.path.exists(out):
        print('skip (done):', out); return
    set_seed(seed)
    model = timm.create_model(model_name, pretrained=True, num_classes=len(SCRIPTS)).to(DEV)
    ttf, etf = make_tfs(model)
    tr, va, te = (META[META.split == s] for s in ('train', 'val', 'test'))
    dl = lambda df, tf, sh: DataLoader(Crops(df, tf), batch_size=CFG['BATCH'], shuffle=sh, num_workers=CFG['WORKERS'], pin_memory=True, drop_last=sh)
    L_tr, L_va, L_te = dl(tr, ttf, True), dl(va, etf, False), dl(te, etf, False)
    w = None
    if CFG['CLASS_BALANCED_LOSS']:
        cnt = tr['script'].value_counts().reindex(SCRIPTS).values.astype(float)
        w = torch.tensor((1 / np.sqrt(cnt)) / (1 / np.sqrt(cnt)).mean(), dtype=torch.float32, device=DEV)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG['LABEL_SMOOTHING'])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['LR'], weight_decay=CFG['WD'])
    total = CFG['EPOCHS'] * len(L_tr); warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda')
    best, best_state, hist = -1, None, []
    for ep in range(CFG['EPOCHS']):
        model.train(); t0 = time.time(); run_loss = 0
        for x, y in L_tr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with torch.autocast('cuda', dtype=torch.float16):
                loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run_loss += loss.item()
        mv = metrics(*predict(model, L_va))
        hist.append({'epoch': ep + 1, 'loss': run_loss / len(L_tr), 'val_acc': mv['acc'], 'val_macro_f1': mv['macro_f1']})
        print(f"{short} seed{seed} ep{ep + 1}: loss {run_loss / len(L_tr):.4f} val acc {mv['acc']:.4f} macro-F1 {mv['macro_f1']:.4f} ({time.time() - t0:.0f}s)")
        if mv['macro_f1'] > best:
            best = mv['macro_f1']; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; best_val = mv
    model.load_state_dict(best_state)
    mt = metrics(*predict(model, L_te))                      # test is touched exactly once, with the selected model
    torch.save(best_state, f'{ROOT}/ckpt/{short}_seed{seed}.pt')
    json.dump({'model': model_name, 'seed': seed, 'cfg': {k: v for k, v in CFG.items()}, 'history': hist,
               'val': best_val, 'test': mt}, open(out, 'w'))
    print(f"TEST {short} seed{seed}: acc {mt['acc']:.4f} macro-F1 {mt['macro_f1']:.4f}")

In [ ]:
for m in CFG['MODELS']:
    for s in CFG['SEEDS']:
        run(m, s)

## 6. Results table (mean and 95% confidence interval over seeds)

In [ ]:
rows, pcs = [], []
for m in CFG['MODELS']:
    a = aggregate(f'{ROOT}/results', m.split('.')[0])
    if a: rows.append(a[0]); pcs.append(a[1])
tab = pd.DataFrame(rows); display(tab)
display(pd.DataFrame(pcs).round(1))
tab.to_csv(f'{ROOT}/results/baseline_table.csv', index=False)

## 7. Parameters, FLOPs and latency of the agent backbones (measured, for Table 2)
FLOPs here count 2 x multiply-accumulates for one 224x224 image (state this convention in the paper). Latency is the median of 200 runs at batch size 1 in fp32 on the GPU shown above. If timm reports an unknown model name, list valid ones with `timm.list_models('*convnext_tiny*', pretrained=True)` and substitute.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

BACKBONES = {'Page agent': 'mobilevit_xs.cvnets_in1k', 'Block agent': 'convnext_tiny.fb_in22k',
             'Line agent': 'vit_small_patch16_224.augreg_in21k', 'Word agent': 'efficientvit_b0.r224_in1k',
             'Flat baseline': 'vit_base_patch16_224.augreg_in21k'}

def efficiency(name, size=224):
    m = timm.create_model(name, pretrained=False, num_classes=len(SCRIPTS)).to(DEV).eval()
    x = torch.randn(1, 3, size, size, device=DEV)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        m(x)
    gflops = fc.get_total_flops() / 1e9
    params = sum(p.numel() for p in m.parameters()) / 1e6
    with torch.no_grad():
        for _ in range(50): m(x)
        torch.cuda.synchronize(); ts = []
        for _ in range(200):
            t = time.perf_counter(); m(x); torch.cuda.synchronize(); ts.append((time.perf_counter() - t) * 1000)
    return round(params, 2), round(gflops, 2), round(float(np.median(ts)), 2)

eff = pd.DataFrame([(k, v) + efficiency(v) for k, v in BACKBONES.items()], columns=['agent', 'timm model', 'params (M)', 'GFLOPs', 'latency (ms)'])
display(eff); eff.to_csv(f'{ROOT}/results/backbone_efficiency.csv', index=False)
print('GPU:', torch.cuda.get_device_name(0))